In [1]:
import pandas as pd
import numpy as np
import os
import sys
#from sklearn.metrics import mean_absolute_error
from scipy.stats import median_abs_deviation

import json

In [2]:
DATASET_MAX_N_OBS = {
    'autompg': 400,
    'breastcancer': 200,
    'fertility': 100,  # Small dataset
    'forest': 500,
    'housing': 500,
    'pendulum': 500,
    'qsar_aquatic_toxicity': 500,
    'servo': 100,      # Small dataset
    'stock': 500,
    'yacht_hydrodynamics': 300 , # Medium dataset
    'ENB2012_data_energy_heating': 768,
    'ENB2012_data_energy_cooling': 768,
    'real_estate': 414,
    'winequality-red': 1599,
    'winequality-white': 4898,
    'airfoil_self_noise': 1503,
    'qsar_fish_toxicity': 908,
    'Combined_Cycle_Power_Plant': 9568,
}

def get_valid_n_obs_list(data_name, requested_n_obs_list):
    """
    Filter n_obs_list based on dataset's maximum available samples
    
    Args:
        data_name: Name of the dataset
        requested_n_obs_list: List of requested sample sizes
        
    Returns:
        Filtered list of valid sample sizes
    """
    max_n_obs = DATASET_MAX_N_OBS.get(data_name, 500)
    valid_list = [n for n in requested_n_obs_list if n <= max_n_obs]
    
    return valid_list

# List of datasets (same as in experiment)
data_names = [
    'fertility', 
    'forest',
    'qsar_aquatic_toxicity', 
    'stock', 
    'yacht_hydrodynamics',
    'real_estate',
    'winequality-red',
    'winequality-white',
    'qsar_fish_toxicity',
    'Combined_Cycle_Power_Plant'
]


In [25]:
def calculate_mse_metrics(data_name, n_obs_list_full = [50,100,200,300,400,500], 
                            results_dir='../results', metrics_dir='./metrics'):
    """
    Calculate MSE for all models across different sample sizes and replications
    
    Args:
        data_name: Dataset name (e.g., 'autompg')
        n_obs_list: List of sample sizes to analyze
        save: Whether to save results to CSV
        results_dir: Directory containing experiment results
        metrics_dir: Directory to save metrics (if None, won't save even if save=True)
    
    Returns:
        DataFrame with columns: data, n_obs, r, base_rf, rf10, rf100, srf_normal_*, srf_hypsec_*
    """
    n_obs_list = get_valid_n_obs_list(data_name, n_obs_list_full)
    # List to store results
    results_list = []
    
    # Define all model names (use original names for reading)
    baseline_model_names = ['rf10', 'rf20', 'rf50', 'rf_100','gp']
    srf_model_names = ['srf_normal_EST_PD','srf_hypsec_EST_PD']
    
    # Iterate through all sample sizes and replications
    for n_obs in n_obs_list:
        for r in range(100):  # 100 replications
            baseline_pred_file = f'{results_dir}/{data_name}/predictions/{data_name}_n{n_obs}_r{r}.csv'
            srf_pred_file = f'{results_dir}/{data_name}/EST_PD_predictions_noCV/{data_name}_n{n_obs}_r{r}.csv'
            
            # Check if file exists
            if not os.path.exists(baseline_pred_file):
                continue
            
            try:
                # Load predictions
                baseline_pred_df = pd.read_csv(baseline_pred_file)
                srf_pred_df = pd.read_csv(srf_pred_file)
                y_test = baseline_pred_df['y_test'].values
                
                # Create result dictionary for this replication
                result = {
                    'data': data_name,
                    'n_obs': n_obs,
                    'r': r
                }
                
                # Calculate MSE for each model
                for model in baseline_model_names:
                    pred_col = f'{model}_pred'
                    
                    if pred_col in baseline_pred_df.columns:
                        y_pred = baseline_pred_df[pred_col].values
                        
                        # Check for NaN values
                        if np.any(np.isnan(y_pred)):
                            result[model] = np.nan
                        else:
                            result[model] = median_abs_deviation(y_pred - y_test)
                    else:
                        result[model] = np.nan
                for srf_model in srf_model_names:
                    pred_col = f'{srf_model}_pred'
                    
                    if pred_col in srf_pred_df.columns:
                        y_pred = srf_pred_df[pred_col].values
                        
                        # Check for NaN values
                        if np.any(np.isnan(y_pred)):
                            result[srf_model] = np.nan
                        else:
                            result[srf_model] = median_abs_deviation(y_pred - y_test)
                    else:
                        result[srf_model] = np.nan
                        
                
                results_list.append(result)
                
            except Exception as e:
                print(f"Error processing {data_name} n={n_obs} r={r}: {e}")
                continue
    
    # Create DataFrame
    results_df = pd.DataFrame(results_list)
    
    # Rename rf_full to rf100
    model_names_display = [m if m != 'rf_100' else 'rf100' for m in baseline_model_names]
    # combine srf_normal_EST_PD and srf_hypsec_EST_PD into baseline_model_names
    results_df.rename(columns={'rf_100': 'rf100'}, inplace=True)
    model_names_display.append('srf_normal_EST_PD')
    model_names_display.append('srf_hypsec_EST_PD')
    # Reorder columns
    cols_order = ['data', 'n_obs', 'r'] + model_names_display
    results_df = results_df[cols_order]
    results_df.rename(columns={
        'data':'Data',
        'rf10':'RF(10)',
        'rf20':'RF(20)',
        'rf50':'RF(50)',
        'rf100': 'RF(100)',
        'gp':'GP',
        'srf_normal_EST_PD':'EST-PD(Norm)',
        'srf_hypsec_EST_PD':'EST-PD(Hypsec)'}, inplace=True)
    
    # Save if requested and metrics_dir is not None
    if metrics_dir is not None:
        save_path = f'{metrics_dir}/{data_name}/{data_name}_MAD.csv'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        results_df.to_csv(save_path, index=False)
        print(f"Saved metrics to: {save_path}")
    
    else:
        return results_df

In [26]:
for data in data_names:
    calculate_mse_metrics(data_name=data,metrics_dir= './metric_MAD')

Saved metrics to: ./metric_MAD/fertility/fertility_MAD.csv
Saved metrics to: ./metric_MAD/forest/forest_MAD.csv
Saved metrics to: ./metric_MAD/qsar_aquatic_toxicity/qsar_aquatic_toxicity_MAD.csv
Saved metrics to: ./metric_MAD/stock/stock_MAD.csv
Saved metrics to: ./metric_MAD/yacht_hydrodynamics/yacht_hydrodynamics_MAD.csv
Saved metrics to: ./metric_MAD/real_estate/real_estate_MAD.csv
Saved metrics to: ./metric_MAD/winequality-red/winequality-red_MAD.csv
Saved metrics to: ./metric_MAD/winequality-white/winequality-white_MAD.csv
Saved metrics to: ./metric_MAD/qsar_fish_toxicity/qsar_fish_toxicity_MAD.csv
Saved metrics to: ./metric_MAD/Combined_Cycle_Power_Plant/Combined_Cycle_Power_Plant_MAD.csv


In [30]:
def combine_all_data_metrics(metric,data_list,metrics_path,save_path=None):
    metric_df = pd.DataFrame()
    for data_name in data_list:
        metric_df = pd.concat([metric_df,pd.read_csv(f'{metrics_path}/{data_name}/{data_name}_{metric}.csv')])

    if save_path is not None:
        metric_df.to_csv(f'{save_path}/{metric}_combined_all_data.csv',index=False)
    else:
        return metric_df

In [32]:
combine_all_data_metrics(metric='MAD',data_list=data_names,
                                        metrics_path= './metric_MAD',
                                        save_path= './metric_MAD')

In [5]:
combined_MAD = pd.read_csv('./metric_MAD/MAD_combined_all_data.csv')

In [6]:
combined_MAD

,Data,n_obs,r,RF(10),RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,fertility,50,0,0.113000,0.106000,0.105633,0.101900,0.130000,0.106955,0.106926
1,fertility,50,1,0.127800,0.124142,0.126444,0.122000,0.130000,0.126199,0.126403
2,fertility,50,2,0.098000,0.118991,0.093167,0.103358,0.123995,0.089859,0.091531
3,fertility,50,3,0.093000,0.113000,0.094200,0.112495,0.119899,0.106772,0.104615
4,fertility,50,4,0.141283,0.121318,0.122753,0.114950,0.130000,0.128246,0.132177
...,...,...,...,...,...,...,...,...,...,...
5295,Combined_Cycle_Power_Plant,500,95,2.874500,2.826154,2.749416,2.729796,3.605561,2.804700,2.784515
5296,Combined_Cycle_Power_Plant,500,96,2.858915,2.783147,2.745395,2.747451,3.416410,2.765300,2.772920
5297,Combined_Cycle_Power_Plant,500,97,2.772500,2.687000,2.643100,2.622950,3.421282,2.682935,2.655395
5298,Combined_Cycle_Power_Plant,500,98,2.775904,2.763583,2.707025,2.676821,3.512856,2.719480,2.700060


In [7]:
def calculate_percentage(df):
    pi_df = pd.DataFrame()
    pi_df['Data'] = df['Data']
    pi_df['N'] = df['N']
    pi_df['P'] = df['P']
    pi_df['n'] = df['n']
    pi_df['r'] = df['r']
    
    pi_df['RF(20)'] = round((df['RF(10)'] - df['RF(20)'])/df['RF(10)']*100,4)
    pi_df['RF(50)'] = round((df['RF(10)'] - df['RF(50)'])/df['RF(10)']*100,4)
    pi_df['RF(100)'] = round((df['RF(10)'] - df['RF(100)'])/df['RF(10)']*100,4)
    pi_df['GP'] = round((df['RF(10)'] - df['GP'])/df['RF(10)']*100,4)

    pi_df['EST-PD(Norm)'] = round((df['RF(10)'] - df['EST-PD(Norm)'])/df['RF(10)']*100,4)
    pi_df['EST-PD(Hypsec)'] = round((df['RF(10)'] - df['EST-PD(Hypsec)'])/df['RF(10)']*100,4)

    return pi_df

In [8]:
def add_data_info(data_names):
    data_info = pd.DataFrame()
    p_list = []
    n_list = []
    for data_name in data_names:
        data = pd.read_csv(f'../../data/{data_name}.csv')
        p_list.append(data.shape[1]-1)
        n_list.append(data.shape[0])
    data_info['Data'] = data_names
    data_info['P'] = p_list
    data_info['N'] = n_list
    return data_info

In [9]:
data_info = add_data_info(data_names)

In [10]:
# combine data_info with best_model_pi
combined_MAD_df = pd.merge(combined_MAD, data_info, on='Data', how='left')
# rename n to n_obs
combined_MAD_df = combined_MAD_df.rename(columns={'n_obs':'n'})
# order the columns
combined_MAD_df = combined_MAD_df[['Data','N','P','n','r','RF(10)','RF(20)','RF(50)','RF(100)','GP','EST-PD(Norm)','EST-PD(Hypsec)']]

In [11]:
combined_MAD_df

,Data,N,P,n,r,RF(10),RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,fertility,100,9,50,0,0.113000,0.106000,0.105633,0.101900,0.130000,0.106955,0.106926
1,fertility,100,9,50,1,0.127800,0.124142,0.126444,0.122000,0.130000,0.126199,0.126403
2,fertility,100,9,50,2,0.098000,0.118991,0.093167,0.103358,0.123995,0.089859,0.091531
3,fertility,100,9,50,3,0.093000,0.113000,0.094200,0.112495,0.119899,0.106772,0.104615
4,fertility,100,9,50,4,0.141283,0.121318,0.122753,0.114950,0.130000,0.128246,0.132177
...,...,...,...,...,...,...,...,...,...,...,...,...
5295,Combined_Cycle_Power_Plant,9568,4,500,95,2.874500,2.826154,2.749416,2.729796,3.605561,2.804700,2.784515
5296,Combined_Cycle_Power_Plant,9568,4,500,96,2.858915,2.783147,2.745395,2.747451,3.416410,2.765300,2.772920
5297,Combined_Cycle_Power_Plant,9568,4,500,97,2.772500,2.687000,2.643100,2.622950,3.421282,2.682935,2.655395
5298,Combined_Cycle_Power_Plant,9568,4,500,98,2.775904,2.763583,2.707025,2.676821,3.512856,2.719480,2.700060


In [25]:
data_name_mapping = {
    'Combined_Cycle_Power_Plant': 'CCPP',
    'qsar_fish_toxicity': 'Qsar Fish Toxicity',
    'real_estate': 'Real Estate',
    'yacht_hydrodynamics': 'Yacht Hydrodynamics',
    'qsar_aquatic_toxicity': 'Qsar Aquatic Toxicity',
    'fertility': 'Fertility',
    'stock': 'Stock',
    'winequality-red': 'Winequality (Red)',
    'winequality-white': 'Winequality (White)',
    'forest': 'Forest'
}
combined_MAD_df['Data'] = combined_MAD_df['Data'].replace(data_name_mapping)

In [26]:
pi_df = calculate_percentage(combined_MAD_df)

In [27]:
pi_df

,Data,N,P,n,r,RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,Fertility,100,9,50,0,6.1947,6.5192,9.8230,-15.0442,5.3494,5.3751
1,Fertility,100,9,50,1,2.8623,1.0613,4.5383,-1.7214,1.2531,1.0934
2,Fertility,100,9,50,2,-21.4199,4.9320,-5.4674,-26.5252,8.3075,6.6005
3,Fertility,100,9,50,3,-21.5054,-1.2903,-20.9626,-28.9232,-14.8083,-12.4894
4,Fertility,100,9,50,4,14.1313,13.1155,18.6387,7.9863,9.2276,6.4452
...,...,...,...,...,...,...,...,...,...,...,...
5295,CCPP,9568,4,500,95,1.6819,4.3515,5.0341,-25.4326,2.4282,3.1305
5296,CCPP,9568,4,500,96,2.6502,3.9707,3.8988,-19.5002,3.2745,3.0079
5297,CCPP,9568,4,500,97,3.0839,4.6673,5.3940,-23.4006,3.2305,4.2238
5298,CCPP,9568,4,500,98,0.4439,2.4813,3.5694,-26.5482,2.0326,2.7322


In [28]:
PIMAD_mean = pi_df.groupby(['Data','N','P']).mean().drop(columns=['r','n']).reset_index()
PIMAD_std = pi_df.groupby(['Data','N','P']).std().drop(columns=['r','n']).reset_index()
pi_df['cases'] = 1
cased_by_n_obs = pi_df.groupby(['Data']).sum().reset_index()
PIMAD_std['cases'] = cased_by_n_obs['cases']
mean_MAD_95CI = pd.DataFrame()
mean_MAD_95CI['Data'] = PIMAD_mean['Data']
mean_MAD_95CI['N'] = PIMAD_mean['N']
mean_MAD_95CI['P'] = PIMAD_mean['P']

mean_MAD_95CI['RF(20)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['RF(20)'], PIMAD_std['RF(20)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['RF(50)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['RF(50)'], PIMAD_std['RF(50)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['RF(100)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['RF(100)'], PIMAD_std['RF(100)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['GP'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['GP'], PIMAD_std['GP']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['EST-PD(Norm)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['EST-PD(Norm)'], PIMAD_std['EST-PD(Norm)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['EST-PD(Hypsec)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['EST-PD(Hypsec)'], PIMAD_std['EST-PD(Hypsec)']/np.sqrt(PIMAD_std['cases']))]

mean_best_model_pi = mean_MAD_95CI.sort_values(by='P', ascending=True)

In [29]:
mean_best_model_pi.to_csv('./metric_MAD/MAD_best_model_pi.csv',index=False)

In [30]:
mean_best_model_pi

,Data,N,P,RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,CCPP,9568,4,2.23 (0.09),3.54 (0.10),3.97 (0.10),-26.10 (1.27),4.68 (0.11),4.87 (0.11)
4,Qsar Fish Toxicity,908,6,2.29 (0.17),3.76 (0.18),4.35 (0.19),-6.14 (0.43),6.55 (0.19),6.55 (0.19)
5,Real Estate,414,6,2.28 (0.26),3.38 (0.28),4.25 (0.29),-76.41 (1.14),-3.56 (0.42),-3.16 (0.42)
9,Yacht Hydrodynamics,308,6,6.04 (1.22),12.32 (1.18),14.86 (1.11),-105.68 (7.56),15.40 (1.14),15.60 (1.14)
3,Qsar Aquatic Toxicity,537,8,2.26 (0.21),3.64 (0.22),3.91 (0.23),-30.24 (0.91),2.42 (0.28),2.50 (0.28)
1,Fertility,100,9,-0.86 (0.89),0.71 (0.86),0.61 (0.93),-11.37 (1.67),6.68 (1.03),6.79 (1.04)
6,Stock,536,11,4.31 (0.22),5.88 (0.24),6.48 (0.24),-115.50 (1.16),12.08 (0.28),12.03 (0.29)
7,Winequality (Red),1599,11,1.42 (0.25),2.94 (0.25),3.46 (0.24),-74.02 (1.57),-0.76 (0.34),-0.63 (0.34)
8,Winequality (White),4898,11,2.48 (0.18),3.98 (0.17),4.51 (0.17),-66.68 (1.12),-0.05 (0.34),-0.08 (0.33)
2,Forest,517,12,2.01 (0.22),4.46 (0.26),5.06 (0.27),47.55 (0.78),22.50 (0.33),22.26 (0.34)


In [31]:
PIMAD_mean = PIMAD_mean.sort_values(by='P', ascending=True)

In [32]:
PIMAD_mean.to_csv('./metric_MAD/MAD_PIMAD_mean.csv',index=False)

In [34]:
mean_MAD_df=combined_MAD_df.groupby(['Data','N','P']).mean().reset_index()
mean_MAD_df.drop(columns=['N','r','n'],inplace=True)
mean_MAD_df = mean_MAD_df.sort_values(by='P', ascending=True)

In [35]:
mean_MAD_df.to_csv('./metric_MAD/MAD_mean_df.csv',index=False)

In [36]:
mean_MAD_df

,Data,P,RF(10),RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,CCPP,4,3.164370,3.091735,3.049923,3.036726,3.969814,3.013687,3.006717
4,Qsar Fish Toxicity,6,0.563544,0.550124,0.541685,0.538298,0.596366,0.525946,0.525978
5,Real Estate,6,3.990832,3.894358,3.847088,3.811979,6.985703,4.115110,4.098046
9,Yacht Hydrodynamics,6,0.579202,0.540549,0.504275,0.493535,1.428850,0.473075,0.469339
3,Qsar Aquatic Toxicity,8,0.726284,0.709072,0.698925,0.696634,0.931041,0.706417,0.706020
1,Fertility,9,0.116014,0.116066,0.114339,0.114207,0.125893,0.106799,0.106709
6,Stock,11,0.004168,0.003976,0.003908,0.003883,0.008823,0.003655,0.003657
7,Winequality (Red),11,0.426768,0.419756,0.413323,0.411200,0.741903,0.428975,0.428413
8,Winequality (White),11,0.516067,0.502843,0.495130,0.492438,0.857444,0.516407,0.516519
2,Forest,12,0.907186,0.887456,0.865067,0.859457,0.475830,0.702179,0.704278
